# CPI VARNN forecast

Use the same VARNN workflow as the export notebook: ARDL-NN-style preprocessing, VAR lag selection on the train split, then an FNN over the selected lag window to forecast `cpi_mom_inflation`.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.api import VAR

try:
    import tensorflow as tf
except ImportError:
    tf = None

MODULE_PATH = Path("rnn-models.py").resolve()
spec = importlib.util.spec_from_file_location("rnn_models", MODULE_PATH)
rnn_models = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = rnn_models
spec.loader.exec_module(rnn_models)


In [ ]:
DATA_PATH = Path("../cpi_forecast_selected_variables.csv")
OUT_DIR = Path("varnn_cpi_results")

data_config = rnn_models.RNNDataConfig(
    date_col="date",
    target="cpi_mom_inflation",
    features=("broad_money", "ppi_qoq", "wti", "gold", "policy_rate", "VNINDEX", "NIKKEI225", "USDVND"),
    freq="MS",
    log_transform=False,
    stationarity=True,
    train_ratio=0.70,
    val_ratio=0.15,
)

VAR_LAG_CRITERION = "aic"
VAR_MAXLAGS = None
MIN_VAR_LAG = 1
HIDDEN_UNITS = 32
EPOCHS = 100
BATCH_SIZE = 32
PATIENCE = 12
SEED = 7


In [ ]:
raw_data, _ = rnn_models.load_processed_like_ardl(DATA_PATH, data_config)
base_train, base_val, base_test = rnn_models.chronological_split(raw_data, data_config.train_ratio, data_config.val_ratio)
transformed, stationarity_screen = rnn_models.apply_train_only_stationarity_transform(
    raw_data,
    base_train,
    enabled=data_config.stationarity,
)

train = transformed.loc[transformed.index.intersection(base_train.index)].copy()
validation = transformed.loc[transformed.index.intersection(base_val.index)].copy()
test = transformed.loc[transformed.index.intersection(base_test.index)].copy()
model_columns = [data_config.target, *data_config.features]
target_idx = model_columns.index(data_config.target)

print("rows", {"raw": len(raw_data), "transformed": len(transformed), "train": len(train), "validation": len(validation), "test": len(test)})
print("columns", model_columns)


In [ ]:
scalers = {col: MinMaxScaler(feature_range=(-1, 1)).fit(train[[col]]) for col in model_columns}

def scale_frame(frame):
    scaled = frame[model_columns].copy()
    for col, scaler in scalers.items():
        scaled[col] = scaler.transform(frame[[col]]).reshape(-1)
    return scaled

def inverse_target(values):
    return scalers[data_config.target].inverse_transform(np.asarray(values).reshape(-1, 1)).reshape(-1)

scaled_train = scale_frame(train)
scaled_val = scale_frame(validation)
scaled_test = scale_frame(test)
scaled_full = pd.concat([scaled_train, scaled_val, scaled_test]).sort_index()


In [ ]:
def choose_var_lag(frame, criterion="aic", maxlags=None, min_lag=1):
    clean_frame = frame.dropna()
    if len(clean_frame) < 8:
        raise ValueError("Need at least 8 train observations to select a VAR lag.")

    non_constant_columns = clean_frame.columns[clean_frame.std(axis=0) > 1e-12].tolist()
    dropped_columns = [col for col in clean_frame.columns if col not in non_constant_columns]
    if data_config.target not in non_constant_columns:
        raise ValueError(f"Target is constant after preprocessing and cannot be used for VAR lag selection: {data_config.target}")
    clean_frame = clean_frame[non_constant_columns]

    if maxlags is None:
        n_obs = len(clean_frame)
        n_vars = clean_frame.shape[1]
        # Keep VAR lag search estimable on short monthly samples.
        maxlags = min(12, max(min_lag, (n_obs - n_vars - 1) // (n_vars + 1)))

    maxlags = int(max(min_lag, min(maxlags, len(clean_frame) - 2)))
    last_error = None
    order_result = None
    maxlags_used = None
    for candidate_maxlags in range(maxlags, min_lag - 1, -1):
        try:
            order_result = VAR(clean_frame).select_order(maxlags=candidate_maxlags)
            maxlags_used = candidate_maxlags
            break
        except Exception as exc:
            last_error = exc

    if order_result is None:
        raise RuntimeError(f"VAR lag selection failed for maxlags <= {maxlags}: {last_error}")

    selected_orders = {key: int(value) for key, value in order_result.selected_orders.items()}
    selected_lag = selected_orders.get(criterion)
    if selected_lag is None:
        raise ValueError(f"Unknown VAR lag criterion: {criterion}")
    selected_lag = max(min_lag, int(selected_lag))

    lag_selection_table = pd.DataFrame(order_result.ics)
    lag_selection_table.insert(0, "lag", range(len(lag_selection_table)))
    lag_selection_table["selected_by"] = ""
    lag_selection_table["var_columns"] = ",".join(non_constant_columns)
    lag_selection_table["dropped_constant_columns"] = ",".join(dropped_columns)
    for key, value in selected_orders.items():
        if value < len(lag_selection_table):
            marker = lag_selection_table.loc[value, "selected_by"]
            lag_selection_table.loc[value, "selected_by"] = f"{marker},{key}" if marker else key

    return selected_lag, selected_orders, maxlags_used, lag_selection_table, non_constant_columns, dropped_columns

P, selected_orders, var_maxlags_used, lag_selection_table, var_lag_columns, dropped_lag_columns = choose_var_lag(
    train[model_columns],
    criterion=VAR_LAG_CRITERION,
    maxlags=VAR_MAXLAGS,
    min_lag=MIN_VAR_LAG,
)
print(
    f"Selected VAR lag P={P} using {VAR_LAG_CRITERION.upper()} "
    f"(searched maxlags={var_maxlags_used}; raw selected orders={selected_orders})"
)
print("VAR lag-selection columns", var_lag_columns)
print("Dropped constant columns for lag selection only", dropped_lag_columns)
lag_selection_table


In [ ]:
def to_sequences_multivariate_by_split(dataset, p, train_index, val_index, test_index):
    x_rows = []
    y_rows = []
    dates = []
    split_labels = []
    for i in range(p, len(dataset)):
        date = dataset.index[i]
        if date in train_index:
            split = "train"
        elif date in val_index:
            split = "validation"
        elif date in test_index:
            split = "test"
        else:
            continue
        x_rows.append(dataset.iloc[i - p:i, 0:dataset.shape[1]].to_numpy(float))
        y_rows.append(dataset.iloc[i:i + 1, 0:dataset.shape[1]].to_numpy(float).reshape(-1))
        dates.append(pd.Timestamp(date))
        split_labels.append(split)
    return np.asarray(x_rows), np.asarray(y_rows), pd.DatetimeIndex(dates), np.asarray(split_labels)

X, y, dates, splits = to_sequences_multivariate_by_split(scaled_full, P, train.index, validation.index, test.index)
train_mask = splits == "train"
val_mask = splits == "validation"
test_mask = splits == "test"

trainX, trainY = X[train_mask], y[train_mask]
valX, valY = X[val_mask], y[val_mask]
testX, testY = X[test_mask], y[test_mask]

shape_table = pd.DataFrame([
    {"split": "all", "selected_lag": P, "lag_criterion": VAR_LAG_CRITERION, "X_shape": tuple(X.shape), "Y_shape": tuple(y.shape)},
    {"split": "train", "selected_lag": P, "lag_criterion": VAR_LAG_CRITERION, "X_shape": tuple(trainX.shape), "Y_shape": tuple(trainY.shape)},
    {"split": "validation", "selected_lag": P, "lag_criterion": VAR_LAG_CRITERION, "X_shape": tuple(valX.shape), "Y_shape": tuple(valY.shape)},
    {"split": "test", "selected_lag": P, "lag_criterion": VAR_LAG_CRITERION, "X_shape": tuple(testX.shape), "Y_shape": tuple(testY.shape)},
])
shape_table


In [ ]:
if tf is None:
    raise ImportError("TensorFlow/Keras is required to train the VARNN model.")

tf.keras.utils.set_random_seed(SEED)

class VARNN(tf.keras.Model):
    def __init__(self, hidden_units, output_dim):
        super(VARNN, self).__init__()
        self.hidden_units = hidden_units
        self.ffnn_model = tf.keras.Sequential([
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(hidden_units, activation="sigmoid"),
            tf.keras.layers.Dense(output_dim),
        ])

    def call(self, inputs):
        return self.ffnn_model(inputs)

varnn_model = VARNN(hidden_units=HIDDEN_UNITS, output_dim=len(model_columns))
varnn_model.compile(optimizer="adam", loss="mse")
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    min_delta=1e-5,
    restore_best_weights=True,
)


In [ ]:
start = time.time()
history = varnn_model.fit(
    trainX,
    trainY,
    verbose=2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(valX, valY),
    callbacks=[early_stopping],
)
training_seconds = time.time() - start
pd.DataFrame(history.history).tail()


In [ ]:
pred_scaled = varnn_model.predict(X, batch_size=BATCH_SIZE, verbose=0)
actual_target = inverse_target(y[:, target_idx])
predicted_target = inverse_target(pred_scaled[:, target_idx])

forecast_df = pd.DataFrame({
    "date": dates,
    "split": splits,
    "actual_raw_transformed": actual_target,
    "predicted_raw_transformed": predicted_target,
})

transform_info = dict(zip(stationarity_screen["Variable"], stationarity_screen["Transform used"]))

def to_level(forecasts):
    actual_level = []
    predicted_level = []
    target_transform = transform_info.get(data_config.target, "level")
    for row in forecasts.itertuples(index=False):
        date = pd.Timestamp(row.date)
        if target_transform == "diff1":
            pos = raw_data.index.get_loc(date)
            if isinstance(pos, slice) or isinstance(pos, np.ndarray):
                raise ValueError(f"Date is not unique in raw_data index: {date}")
            if pos <= 0:
                raise ValueError(f"Cannot invert diff1 for first timestamp: {date}")
            previous_base = float(raw_data.iloc[pos - 1][data_config.target])
            actual_base = previous_base + float(row.actual_raw_transformed)
            predicted_base = previous_base + float(row.predicted_raw_transformed)
        else:
            actual_base = float(row.actual_raw_transformed)
            predicted_base = float(row.predicted_raw_transformed)
        if data_config.log_transform:
            actual_base = float(np.exp(actual_base))
            predicted_base = float(np.exp(predicted_base))
        actual_level.append(actual_base)
        predicted_level.append(predicted_base)
    return np.asarray(actual_level), np.asarray(predicted_level)

forecast_df["actual_level"], forecast_df["predicted_level"] = to_level(forecast_df)
forecast_df.tail()


In [ ]:
def evaluate_forecast(actual, predicted):
    actual_arr = np.asarray(actual, dtype=float)
    pred_arr = np.asarray(predicted, dtype=float)
    rmse = float(np.sqrt(mean_squared_error(actual_arr, pred_arr)))
    mae = float(mean_absolute_error(actual_arr, pred_arr))
    denominator = np.where(np.abs(actual_arr) < 1e-12, np.nan, np.abs(actual_arr))
    mape = float(np.nanmean(np.abs((actual_arr - pred_arr) / denominator)) * 100.0)
    return pd.Series({"RMSE": rmse, "MAE": mae, "MAPE": mape})

metric_specs = [
    ("raw_transformed", "actual_raw_transformed", "predicted_raw_transformed"),
    ("level", "actual_level", "predicted_level"),
]
metric_rows = []
for scale, actual_col, predicted_col in metric_specs:
    rows = (
        forecast_df.groupby("split", sort=False)
        .apply(lambda g: evaluate_forecast(g[actual_col], g[predicted_col]))
        .reset_index()
    )
    rows.insert(0, "model", "VARNN")
    rows.insert(2, "target", data_config.target)
    rows.insert(3, "scale", scale)
    metric_rows.append(rows)

metrics_table = pd.concat(metric_rows, ignore_index=True)
metrics_wide = metrics_table.pivot(index=["model", "target", "split"], columns="scale", values=["RMSE", "MAE", "MAPE"])
metrics_wide.columns = [f"{metric}_{scale}" for metric, scale in metrics_wide.columns]
metrics_wide = metrics_wide.reset_index()
metrics_table


In [ ]:
metrics_wide


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
forecast_df.to_csv(OUT_DIR / "varnn_forecasts.csv", index=False)
metrics_table.to_csv(OUT_DIR / "varnn_metrics.csv", index=False)
metrics_wide.to_csv(OUT_DIR / "varnn_metrics_wide.csv", index=False)
shape_table.to_csv(OUT_DIR / "varnn_input_shapes.csv", index=False)
lag_selection_table.to_csv(OUT_DIR / "varnn_var_lag_selection.csv", index=False)
pd.DataFrame(history.history).to_csv(OUT_DIR / "varnn_training_history.csv", index=False)
stationarity_screen.to_csv(OUT_DIR / "varnn_stationarity_screen_train_only.csv", index=False)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=False)
for ax, split in zip(axes, ["train", "validation", "test"]):
    split_df = forecast_df[forecast_df["split"].eq(split)].sort_values("date")
    ax.plot(split_df["date"], split_df["actual_level"], label="Actual level", linewidth=2)
    ax.plot(split_df["date"], split_df["predicted_level"], label="Predicted level", linewidth=2)
    ax.set_title(f"VARNN actual vs predicted - {split}")
    ax.set_xlabel("Time")
    ax.set_ylabel(f"{data_config.target} level")
    ax.grid(True, alpha=0.25)
    ax.legend()

fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUT_DIR / "varnn_actual_vs_predicted_train_val_test.png", dpi=160)
plt.show()


In [ ]:
stationarity_screen
